# Självstudier – Föreläsning 4: Dataanalys och visualisering med pandas

Efter de tre första föreläsningarna kan vi skriva Pythonkod själva. Nu använder vi ett bibliotek som är gjort för **tabulär data**: pandas.

Efter materialet bör du kunna:

- skapa och läsa en `DataFrame`,
- välja kolumner och filtrera rader,
- skapa nya kolumner,
- sammanfatta och gruppera data,
- skapa enkla linje-, scatter- och histogramdiagram,
- tolka vad ett diagram visar.

## 1. Data som vårt eget program producerar

Vi börjar med kaströrelsen från tidigare kursmaterial. Funktionen returnerar två listor: horisontell och vertikal position.

In [ ]:
import math
import numpy as np

def kaströrelse(initial_hastighet, vinkel, tid, antal_steg):
    vinkel_radianer = math.radians(vinkel)
    initial_hastighet_x = initial_hastighet * math.cos(vinkel_radianer)
    initial_hastighet_y = initial_hastighet * math.sin(vinkel_radianer)

    tider = np.linspace(0, tid, antal_steg)
    x_positions = []
    y_positions = []

    for t in tider:
        x = initial_hastighet_x * t
        y = initial_hastighet_y * t - 0.5 * 9.81 * t**2

        if y < 0:
            break

        x_positions.append(x)
        y_positions.append(y)

    return x_positions, y_positions

In [ ]:
x_positions, y_positions = kaströrelse(20, 45, 10, 200)

print(x_positions[:5])
print(y_positions[:5])

De två listorna hör ihop: `x_positions[10]` och `y_positions[10]` beskriver samma punkt.

När data består av flera kolumner blir en tabell ofta tydligare än separata listor.

## 2. En DataFrame

`pandas` brukar importeras som `pd`. En `DataFrame` är en tabell med namngivna kolumner.

In [ ]:
import pandas as pd

trajectory = pd.DataFrame({
    "x": x_positions,
    "y": y_positions
})

trajectory.head()

Några snabba sätt att orientera sig:

In [ ]:
print("Form:", trajectory.shape)
print("Kolumner:", list(trajectory.columns))
print()
print(trajectory.head())

## 3. Välj en kolumn

Hakparenteser med ett kolumnnamn väljer kolumnen.

In [ ]:
trajectory["y"].head()

En enskild kolumn är en pandas `Series`. Flera kolumner inom dubbla hakparenteser ger en ny DataFrame.

In [ ]:
print(type(trajectory["y"]))
print(type(trajectory[["x", "y"]]))

## 4. Filtrera rader

Precis som en vanlig jämförelse ger ett booleskt värde, ger en jämförelse med en hel kolumn en serie av `True` och `False`.

In [ ]:
trajectory["y"] > 10

Den booleska serien kan användas för att välja de rader där villkoret är sant.

In [ ]:
trajectory[trajectory["y"] > 10].head()

Det motsvarar ungefär idén från föreläsning 2: **loopa + if + filtrera**, men pandas uttrycker den på hela kolumnen samtidigt.

## 5. Sammanfatta och hitta extrema värden

In [ ]:
max_height = trajectory["y"].max()
print("Maxhöjd:", max_height)

trajectory[trajectory["y"] == max_height]

### Prova själv

Ta reda på minsta och största `x`-värdet.

In [ ]:
# Skriv din kod här

## 6. Skapa en ny kolumn

Operationer mellan kolumner görs rad för rad utan att vi själva skriver en loop.

In [ ]:
trajectory["distance_from_start"] = (
    trajectory["x"]**2 + trajectory["y"]**2
) ** 0.5

trajectory.head()

## 7. Från tabell till bild

En bild kan göra strukturen i data mycket lättare att se.

In [ ]:
import matplotlib.pyplot as plt

trajectory.plot(x="x", y="y")
plt.xlabel("Horisontell position (m)")
plt.ylabel("Vertikal position (m)")
plt.title("Kaströrelse")
plt.show()

Diagrammet innehåller inte mer information än tabellen, men rörelsens form blir omedelbart synlig.

## 8. Data från en CSV-fil

I kursmiljön används CO₂-filen `co2-mm-mlo-0-2014.csv`. `read_csv()` läser filen direkt till en DataFrame.

In [ ]:
co2data = pd.read_csv("co2-mm-mlo-0-2014.csv")
co2data.head()

Börja alltid med att undersöka datan innan du analyserar den.

In [ ]:
print(co2data.shape)
print(list(co2data.columns))
co2data.head()

## 9. Begränsa analysen

Vi kan skapa en ny DataFrame med bara de rader och kolumner vi vill arbeta med.

In [ ]:
co2 = co2data[co2data["year"] >= 1900].copy()

co2[[
    "year",
    "data_mean_global",
    "data_mean_nh",
    "data_mean_sh"
]].head()

## 10. Flera villkor

I pandas används `&` för "och" mellan två kolumnvillkor. Sätt varje villkor inom parentes.

In [ ]:
co2[
    (co2["year"] >= 1950) &
    (co2["data_mean_global"] > 310)
].head()

Vanliga Python-ordet `and` fungerar inte på samma sätt för hela pandas-kolumner.

## 11. Nya analyskolumner

Vi kan skapa en kolumn som inte fanns i originalfilen. Här beräknar vi skillnaden mellan norra och södra halvklotet.

In [ ]:
co2["nh_minus_sh"] = (
    co2["data_mean_nh"] - co2["data_mean_sh"]
)

co2[["year", "nh_minus_sh"]].head()

## 12. Enkel sammanfattande statistik

In [ ]:
print("min:", co2["data_mean_global"].min())
print("max:", co2["data_mean_global"].max())
print("mean:", co2["data_mean_global"].mean())

## 13. `groupby`: sammanfatta grupper

Vi skapar först en kolumn för årtionde och beräknar sedan medelvärdet inom varje grupp.

In [ ]:
co2["decade"] = (co2["year"] // 10) * 10

decade_means = co2.groupby("decade")["data_mean_global"].mean()
decade_means

Tänk på `groupby` som:

> dela upp raderna i grupper → gör samma sammanfattning för varje grupp.

## 14. Sortera

`sort_values()` kan användas för att hitta rader med särskilt höga eller låga värden.

In [ ]:
co2.sort_values("data_mean_global", ascending=False).head()

## 15. Linjediagram

Linjediagram passar när x-axeln har en naturlig ordning, särskilt tid.

In [ ]:
co2.plot(x="year", y="data_mean_global")
plt.xlabel("År")
plt.ylabel("CO₂ (ppm)")
plt.title("Global atmosfärisk CO₂")
plt.show()

### Fråga

Vad ser du i figuren som var svårare att se i tabellen? Skriv gärna en mening i en markdown-cell.

## 16. Flera serier i samma diagram

In [ ]:
co2.plot(
    x="year",
    y=["data_mean_global", "data_mean_nh", "data_mean_sh"]
)
plt.xlabel("År")
plt.ylabel("CO₂ (ppm)")
plt.title("CO₂ globalt och per halvklot")
plt.show()

## 17. Scatterplot

Scatterplot används ofta för att undersöka relationen mellan två variabler.

In [ ]:
co2.plot.scatter(
    x="data_mean_nh",
    y="data_mean_sh"
)
plt.xlabel("Norra halvklotet")
plt.ylabel("Södra halvklotet")
plt.title("Relation mellan halvkloten")
plt.show()

En tydlig struktur i ett scatterplot betyder att variablerna verkar samvariera. Det betyder **inte automatiskt** att den ena orsakar den andra.

## 18. Histogram

Ett histogram visar hur värden i en variabel fördelar sig.

In [ ]:
co2["nh_minus_sh"].hist()
plt.xlabel("Skillnad NH - SH (ppm)")
plt.ylabel("Antal observationer")
plt.title("Fördelning av skillnaden mellan halvkloten")
plt.show()

## 19. Saknade värden

Verkliga data kan sakna observationer. En snabb kontroll är:

In [ ]:
co2data.isna().sum()

Det finns ingen universell regel för vad man ska göra med saknade värden. Ibland tar man bort rader, ibland fyller man i värden, och ibland behöver man förstå **varför** data saknas innan man gör något alls.

## 20. Arbetsflödet för dataanalys

Ett användbart arbetssätt är:

**läs in → inspektera → välj/filtrera → skapa variabler → sammanfatta → visualisera → tolka**

Nästa föreläsning bygger vidare på exakt detta. Skillnaden är att vi då låter en modell lära sig ett samband från data.

# Självtest

1. Vad är en `DataFrame`?
2. Hur väljer man kolumnen `temperature` ur `df`?
3. Vad gör `df[df["temperature"] > 20]`?
4. Varför används `&` och parenteser när två pandas-villkor kombineras?
5. Vad gör `groupby()` på en övergripande nivå?
6. När passar ett linjediagram bättre än ett scatterplot?
7. Vad visar ett histogram?
8. Varför bör vi inte automatiskt fylla alla saknade värden med medelvärdet?

## Inför laboration 4

Laboration 4 bygger på hela arbetsflödet ovan. Du bör kunna:

- läsa och inspektera en DataFrame,
- filtrera med ett eller flera villkor,
- skapa nya kolumner,
- använda enkla sammanfattningar och `groupby`,
- skapa ett diagram och skriva några meningar om vad det visar.

Det viktiga är alltså inte bara att få fram en figur, utan att **använda figuren för att besvara en fråga**.

# Facit till självtestet

<details>
<summary><strong>Klicka här när du har försökt själv</strong></summary>

1. En tabelliknande datastruktur med namngivna kolumner och rader.
2. `df["temperature"]`.
3. Väljer de rader där kolumnen `temperature` är större än 20.
4. Pandas arbetar med en hel serie booleska värden; `&` kombinerar dem elementvis och parenteserna avgränsar villkoren.
5. Delar upp data i grupper och gör det möjligt att sammanfatta varje grupp separat.
6. När x-axeln har en naturlig ordning, särskilt tid.
7. Hur observationerna i en numerisk variabel är fördelade över intervall.
8. Därför att orsaken till saknade data och vilken behandling som är rimlig beror på sammanhanget.

</details>

## Sammanfattning

Pandas gör det möjligt att uttrycka många dataoperationer på hela tabeller och kolumner i stället för att själv skriva loopar. Visualisering hjälper oss sedan att se mönster som är svåra att upptäcka i en tabell.

På nästa träff går vi från **beskriva data** till **lära en modell från data**.